# Progetto MIGA 2025/2026 — Sistema di Raccomandazione e Sentiment Analysis
## Notebook unico — Dataset Amazon Reviews 2023, categoria Movies_and_TV (Film & TV)

Questo notebook raccoglie l'intero progetto di **livello avanzato** in un solo file, in tre parti:

1. **Parte 1 — Progetto Base**: Collaborative Filtering (K-NN ottimale, filling, K-Means + cosine, top-N, Matrix Factorization).
2. **Parte 2 — Progetto Intermedio**: Content-Based (NLP, embedding TF-IDF e transformer, K-NN per utente, confronti critici).
3. **Parte 3 — Progetto Avanzato**: Sentiment Analysis (classificatori scikit-learn + bonus LLM).

**Come eseguire (Google Colab):** attiva la **GPU** (Runtime > Cambia tipo di runtime > T4), esegui la cella di installazione **una sola volta**, poi tutte le celle in ordine dall'alto verso il basso. La categoria si cambia nelle celle di configurazione di ciascuna parte (`CATEGORY`).

> Nota: Movies_and_TV e' enorme, quindi i dati vengono caricati **in streaming** su un campione. Le tre parti caricano i dati in modo indipendente: se vuoi puoi abbassare `MAX_REVIEWS` per andare piu veloce in fase di test.

## Installazione (eseguire una sola volta)

In [ ]:
!pip -q install scikit-surprise datasets sentence-transformers transformers nltk scikit-learn pandas numpy matplotlib seaborn

---
# PARTE 1 — Progetto Base (Collaborative Filtering)

## 0. Setup ambiente

`scikit-surprise` su Colab a volte richiede una versione di NumPy compatibile: se l'import fallisce, riavvia il runtime dopo l'installazione (Runtime > Restart).

In [ ]:
# Installazione librerie (eseguire una volta su Colab)
!pip -q install scikit-surprise datasets pandas numpy matplotlib seaborn scikit-learn
# Se 'surprise' da' errore di import dopo l'installazione: Runtime > Restart session, poi ri-esegui dalle celle sotto.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from surprise import Dataset, Reader, KNNWithMeans, KNNBasic, SVD, accuracy
from surprise.model_selection import GridSearchCV, train_test_split, cross_validate

from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup OK")

## Configurazione

- `CATEGORY`: categoria merceologica da analizzare (default consigliato: `All_Beauty`).
- `MIN_USER_RATINGS` / `MIN_ITEM_RATINGS`: soglie di filtro per ridurre la sparsita e rendere il Collaborative Filtering significativo e computazionalmente trattabile (discutere questa scelta nel report: trade-off copertura vs densita / cold-start).

In [ ]:
CATEGORY = "Movies_and_TV"   # categoria Film & TV
MIN_USER_RATINGS = 5         # tieni utenti con almeno N recensioni
MIN_ITEM_RATINGS = 5         # tieni item con almeno N recensioni
RATING_SCALE = (1, 5)
TOP_N = 10                   # quante raccomandazioni per utente

# Movies_and_TV e' enorme: lavoriamo su un campione in streaming per non saturare la RAM di Colab
STREAMING = True
MAX_REVIEWS = 800000         # dimensione del campione iniziale (alza se hai RAM)
# Il "core": teniamo gli utenti e gli item piu attivi per avere una matrice densa e trattabile
USERS_CAP = 3000
ITEMS_CAP = 2000

## Step 0 — Caricamento dati (solo user reviews)

Per il Progetto Base si usa **solo il file delle recensioni** (`user reviews`). Carichiamo da HuggingFace la config `raw_review_<CATEGORY>`.

In [ ]:
from datasets import load_dataset

def load_reviews(category, max_rows=MAX_REVIEWS, streaming=STREAMING):
    # Le nuove versioni di datasets non eseguono lo script del repo:
    # leggiamo direttamente il file .jsonl grezzo col builder "json".
    url = f"hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/review_categories/{category}.jsonl"
    ds = load_dataset("json", data_files=url, split="train", streaming=streaming)
    if streaming:
        ds = ds.shuffle(seed=42, buffer_size=50000)
        rows = []
        for ex in ds:
            rows.append(ex)
            if len(rows) >= max_rows:
                break
        df_ = pd.DataFrame(rows)
    else:
        df_ = ds.to_pandas()
    # nel jsonl grezzo i campi si chiamano sort_timestamp / helpful_votes
    return df_.rename(columns={"sort_timestamp": "timestamp", "helpful_votes": "helpful_vote"})

df = load_reviews(CATEGORY)
print("Shape campione:", df.shape)
df.head(3)

In [ ]:
# Teniamo solo le colonne utili per il CF + qualche colonna per la EDA
cols = ["rating", "title", "text", "asin", "parent_asin", "user_id",
        "timestamp", "verified_purchase", "helpful_vote"]
df = df[[c for c in cols if c in df.columns]].copy()

# Usiamo parent_asin come identificatore di item (la traccia nota che asin storici = parent_asin)
df["item_id"] = df["parent_asin"].fillna(df["asin"])
df = df.dropna(subset=["user_id", "item_id", "rating"])
df["rating"] = df["rating"].astype(float)
print("Shape pulita:", df.shape)

## Step 1 — Analisi Esplorativa (EDA)

Statistiche descrittive, distribuzioni e analisi di correlazione.

In [ ]:
print("Recensioni totali :", len(df))
print("Utenti unici       :", df["user_id"].nunique())
print("Item unici         :", df["item_id"].nunique())
print("Periodo            :", pd.to_datetime(df["timestamp"], unit="ms").min(),
      "->", pd.to_datetime(df["timestamp"], unit="ms").max())
display(df["rating"].describe())

In [ ]:
# Distribuzione dei rating
plt.figure(figsize=(7,4))
ax = sns.countplot(x="rating", data=df, color="#7AC142")
ax.set_title(f"Distribuzione dei rating — {CATEGORY}")
ax.set_xlabel("Rating"); ax.set_ylabel("Conteggio")
for p in ax.patches:
    ax.annotate(f"{p.get_height():,}", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

print("Rating medio:", round(df["rating"].mean(), 3))
print("% rating >= 4:", round((df["rating"]>=4).mean()*100, 1), "%  (tipico forte sbilanciamento positivo)")

In [ ]:
# Quante recensioni per utente e per item (long tail tipica)
rpu = df.groupby("user_id").size()
rpi = df.groupby("item_id").size()

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].hist(rpu, bins=50, color="#4A90D9"); axes[0].set_yscale("log")
axes[0].set_title("Recensioni per utente"); axes[0].set_xlabel("# recensioni"); axes[0].set_ylabel("# utenti (log)")
axes[1].hist(rpi, bins=50, color="#9B59B6"); axes[1].set_yscale("log")
axes[1].set_title("Recensioni per item"); axes[1].set_xlabel("# recensioni"); axes[1].set_ylabel("# item (log)")
plt.tight_layout(); plt.show()

n_pairs = df["user_id"].nunique() * df["item_id"].nunique()
sparsity = 1 - len(df)/n_pairs
print(f"Sparsita della matrice utente-item: {sparsity*100:.4f}%  (vicino al 100% = molto sparsa)")

In [ ]:
# Verified purchase e helpful vote
if "verified_purchase" in df.columns:
    print(df["verified_purchase"].value_counts(normalize=True).rename("quota"))
if "helpful_vote" in df.columns:
    print("\nHelpful vote — describe:")
    display(df["helpful_vote"].describe())

In [ ]:
# Andamento del rating medio nel tempo
df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
monthly = df.set_index("date")["rating"].resample("ME").mean()
plt.figure(figsize=(10,3.5))
monthly.plot(color="#7AC142")
plt.title(f"Rating medio mensile — {CATEGORY}"); plt.ylabel("Rating medio"); plt.xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
# Analisi di correlazione tra variabili numeriche
df["text_len"] = df["text"].fillna("").str.len()
num_cols = [c for c in ["rating", "helpful_vote", "verified_purchase", "text_len"] if c in df.columns]
corr_df = df[num_cols].copy()
if "verified_purchase" in corr_df:
    corr_df["verified_purchase"] = corr_df["verified_purchase"].astype(int)
corr = corr_df.corr()
plt.figure(figsize=(5.5,4.5))
sns.heatmap(corr, annot=True, cmap="RdYlGn", center=0, fmt=".2f")
plt.title("Matrice di correlazione")
plt.tight_layout(); plt.show()
display(corr)

### Filtro di densita

Per rendere il CF significativo (e la matrice trattabile) teniamo solo utenti e item con un numero minimo di interazioni. Iteriamo il filtro finche stabile, perche rimuovere utenti puo far scendere item sotto soglia e viceversa.

In [ ]:
def filter_min(df, min_u, min_i):
    prev = None
    cur = df.copy()
    while prev is None or len(cur) != prev:
        prev = len(cur)
        u_ok = cur.groupby("user_id").size()
        cur = cur[cur["user_id"].isin(u_ok[u_ok >= min_u].index)]
        i_ok = cur.groupby("item_id").size()
        cur = cur[cur["item_id"].isin(i_ok[i_ok >= min_i].index)]
    return cur

df_cf = filter_min(df, MIN_USER_RATINGS, MIN_ITEM_RATINGS)

# Movies e' grande e sparso lato utente: teniamo il "core" piu attivo (utenti + item)
# per ottenere una matrice densa e trattabile (logica "most relevant users/products").
top_u = df_cf.groupby("user_id").size().sort_values(ascending=False).head(USERS_CAP).index
top_i = df_cf.groupby("item_id").size().sort_values(ascending=False).head(ITEMS_CAP).index
df_cf = df_cf[df_cf["user_id"].isin(top_u) & df_cf["item_id"].isin(top_i)]
df_cf = filter_min(df_cf, MIN_USER_RATINGS, MIN_ITEM_RATINGS)

# Una sola recensione per coppia (user,item) per evitare duplicati nella matrice
df_cf = df_cf.sort_values("timestamp").drop_duplicates(["user_id", "item_id"], keep="last")
print("Core CF:", df_cf.shape,
      "| utenti:", df_cf["user_id"].nunique(), "| item:", df_cf["item_id"].nunique())

## Step 2 — Configurazione ottimale K-NN (Surprise)

Testiamo le combinazioni richieste: **metrica di similarita** (msd / cosine / pearson), **valore di K**, **user-based vs item-based**. Selezioniamo la configurazione con il **RMSE/MSE** minimo in cross-validation.

In [ ]:
reader = Reader(rating_scale=RATING_SCALE)
data = Dataset.load_from_df(df_cf[["user_id", "item_id", "rating"]], reader)

param_grid = {
    "k": [10, 20, 40],
    "sim_options": {
        "name": ["msd", "cosine", "pearson"],
        "user_based": [True, False],
    },
}

gs = GridSearchCV(KNNWithMeans, param_grid, measures=["rmse", "mse"], cv=3, n_jobs=-1)
gs.fit(data)

print("Miglior RMSE :", round(gs.best_score["rmse"], 4))
print("Miglior MSE  :", round(gs.best_score["mse"], 4))
print("Config        :", gs.best_params["rmse"])

In [ ]:
# Tabella completa dei risultati della grid search (utile per il report)
res = pd.DataFrame(gs.cv_results)
show_cols = ["param_k", "param_sim_options", "mean_test_rmse", "mean_test_mse", "rank_test_rmse"]
res_show = res[show_cols].sort_values("mean_test_rmse").reset_index(drop=True)
display(res_show)

In [ ]:
# Grafico RMSE vs K per ciascuna combinazione (similarita x user/item)
res["sim_name"] = res["param_sim_options"].apply(lambda d: d["name"])
res["ub"] = res["param_sim_options"].apply(lambda d: "user-based" if d["user_based"] else "item-based")
res["combo"] = res["sim_name"] + " | " + res["ub"]

plt.figure(figsize=(8,5))
for combo, g in res.groupby("combo"):
    g = g.sort_values("param_k")
    plt.plot(g["param_k"], g["mean_test_rmse"], marker="o", label=combo)
plt.xlabel("K (numero di vicini)"); plt.ylabel("RMSE (CV)")
plt.title("K-NN: RMSE al variare di K, similarita e user/item based")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## Step 3 — Filling della matrice di rating (configurazione ottimale)

Addestriamo il K-NN migliore su tutto il dataset e prediciamo i rating mancanti, ottenendo la matrice densa utente-item.

In [ ]:
best_params = gs.best_params["rmse"]
algo_knn = KNNWithMeans(k=best_params["k"], sim_options=best_params["sim_options"])

trainset = data.build_full_trainset()
algo_knn.fit(trainset)

# Predizione delle coppie NON osservate (anti-testset)
anti = trainset.build_anti_testset()
print("Coppie da predire:", len(anti))   # se enorme, alza le soglie MIN_* o campiona
pred_missing = algo_knn.test(anti)

# Rating osservati
obs = df_cf[["user_id", "item_id", "rating"]].rename(
    columns={"user_id":"u", "item_id":"i", "rating":"r"})
# Rating predetti
pred_df = pd.DataFrame([(p.uid, p.iid, p.est) for p in pred_missing], columns=["u","i","r"])

full_long = pd.concat([obs, pred_df], ignore_index=True)
rating_matrix = full_long.pivot_table(index="u", columns="i", values="r")
print("Matrice riempita:", rating_matrix.shape, "| valori mancanti residui:", rating_matrix.isna().sum().sum())
rating_matrix.iloc[:5, :5]

## Step 4 — Segmentazione utenti con K-MEANS + cosine similarity

Ogni utente e rappresentato dal suo vettore di rating (riga della matrice riempita). Per usare la **cosine similarity** con K-Means (che lavora in distanza euclidea), normalizziamo i vettori a norma L2: su vettori L2-normalizzati la distanza euclidea e monotona rispetto alla cosine similarity. Scegliamo K con **Elbow (WCSS)** e **Silhouette**.

In [ ]:
X = rating_matrix.fillna(rating_matrix.mean()).values
X_norm = normalize(X, norm="l2")   # cosine -> euclidea su vettori normalizzati

wcss, sil = [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, max_iter=300, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_norm)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_norm, labels, metric="cosine"))

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(list(K_range), wcss, marker="o", color="#4A90D9"); ax[0].set_title("Elbow (WCSS)")
ax[0].set_xlabel("K"); ax[0].set_ylabel("WCSS")
ax[1].plot(list(K_range), sil, marker="o", color="#7AC142"); ax[1].set_title("Silhouette (cosine)")
ax[1].set_xlabel("K"); ax[1].set_ylabel("Silhouette")
plt.tight_layout(); plt.show()

best_k_clusters = list(K_range)[int(np.argmax(sil))]
print("K suggerito dalla Silhouette:", best_k_clusters)

In [ ]:
# Clustering finale + profilazione dei segmenti
km = KMeans(n_clusters=best_k_clusters, init="k-means++", n_init=10, random_state=RANDOM_STATE)
user_clusters = pd.Series(km.fit_predict(X_norm), index=rating_matrix.index, name="cluster")

prof = pd.DataFrame({"cluster": user_clusters})
prof["rating_medio"] = rating_matrix.mean(axis=1).values
summary = prof.groupby("cluster").agg(n_utenti=("rating_medio","size"),
                                      rating_medio=("rating_medio","mean"))
display(summary)

## Step 5 — Top-N item consigliati per ogni utente

Per ogni utente ordiniamo gli item **non ancora valutati** in base al rating predetto e prendiamo i primi N.

In [ ]:
rated_by_user = df_cf.groupby("user_id")["item_id"].agg(set).to_dict()

def top_n_for_user(user, n=TOP_N):
    if user not in rating_matrix.index:
        return []
    seen = rated_by_user.get(user, set())
    row = rating_matrix.loc[user]
    candidates = row[~row.index.isin(seen)].sort_values(ascending=False)
    return list(candidates.head(n).items())

example_user = rating_matrix.index[0]
print(f"Top-{TOP_N} raccomandazioni per l'utente {example_user}:")
for item, est in top_n_for_user(example_user):
    print(f"  {item}  ->  rating predetto {est:.2f}")

## Step 6 — Matrix Factorization (SVD) e confronto con K-NN

Confrontiamo la configurazione K-NN ottimale con la **Matrix Factorization (SVD)** in termini di MSE e RMSE (cross-validation).

In [ ]:
cv_knn = cross_validate(algo_knn, data, measures=["RMSE", "MSE"], cv=5, n_jobs=-1, verbose=False)
cv_svd = cross_validate(SVD(random_state=RANDOM_STATE), data, measures=["RMSE", "MSE"], cv=5, n_jobs=-1, verbose=False)

compare = pd.DataFrame({
    "Modello": ["K-NN (ottimale)", "Matrix Factorization (SVD)"],
    "RMSE": [cv_knn["test_rmse"].mean(), cv_svd["test_rmse"].mean()],
    "MSE":  [cv_knn["test_mse"].mean(),  cv_svd["test_mse"].mean()],
})
display(compare)

ax = compare.set_index("Modello")[["RMSE","MSE"]].plot(kind="bar", figsize=(7,4),
        color=["#7AC142","#4A90D9"], rot=0)
ax.set_title("Confronto K-NN vs Matrix Factorization")
ax.set_ylabel("Errore (CV media)")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.tight_layout(); plt.show()

## Conclusioni Progetto Base

Sintesi da riportare nel report (riempire con i tuoi numeri):

- **Dataset / EDA:** sparsita, sbilanciamento verso rating alti, long-tail di utenti/item, correlazioni deboli tra rating e helpful_vote / lunghezza testo.
- **K-NN ottimale:** indica metrica di similarita, K e user/item based vincenti + RMSE/MSE.
- **Clustering:** numero di segmenti scelto (Elbow/Silhouette) e descrizione qualitativa dei cluster.
- **Top-N:** esempio di lista raccomandata.
- **K-NN vs Matrix Factorization:** quale generalizza meglio e perche (la MF tende a gestire meglio la sparsita catturando fattori latenti).

> Prossimi notebook: **Intermedio** (content-based con TF-IDF e transformers) e **Avanzato** (sentiment analysis).

---
# PARTE 2 — Progetto Intermedio (Content-Based)

In [ ]:
import re, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
for pkg in ["stopwords", "wordnet", "omw-1.4", "punkt", "punkt_tab"]:
    try: nltk.download(pkg, quiet=True)
    except Exception: pass

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup OK")

In [ ]:
CATEGORY = "Movies_and_TV"  # stessa categoria del Notebook 1
MIN_USER_RATINGS = 5
MIN_ITEM_RATINGS = 5
K_NEIGHBORS = 10            # K per il content-based KNN
RATING_SCALE = (1, 5)

STREAMING = True
MAX_REVIEWS = 800000
USERS_CAP = 3000
ITEMS_CAP = 2000

## Step 0 — Caricamento recensioni + metadati prodotti

Per il content-based servono entrambi i file: le recensioni (per i rating) e i metadati (per il testo dei prodotti).

In [ ]:
from datasets import load_dataset

def load_reviews(category, max_rows=MAX_REVIEWS, streaming=STREAMING):
    # Le nuove versioni di datasets non eseguono lo script del repo:
    # leggiamo direttamente il file .jsonl grezzo col builder "json".
    url = f"hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/review_categories/{category}.jsonl"
    ds = load_dataset("json", data_files=url, split="train", streaming=streaming)
    if streaming:
        ds = ds.shuffle(seed=42, buffer_size=50000)
        rows = []
        for ex in ds:
            rows.append(ex)
            if len(rows) >= max_rows:
                break
        df_ = pd.DataFrame(rows)
    else:
        df_ = ds.to_pandas()
    # nel jsonl grezzo i campi si chiamano sort_timestamp / helpful_votes
    return df_.rename(columns={"sort_timestamp": "timestamp", "helpful_votes": "helpful_vote"})

# Recensioni (campione in streaming)
rev = load_reviews(CATEGORY)
rev["item_id"] = rev["parent_asin"].fillna(rev["asin"])
rev = rev.dropna(subset=["user_id", "item_id", "rating"])
rev["rating"] = rev["rating"].astype(float)
print("Recensioni (campione):", rev.shape)
rev.head(3)

In [ ]:
# Filtro di densita + core piu attivo (coerente col Notebook 1)
def filter_min(df, min_u, min_i):
    prev = None; cur = df.copy()
    while prev is None or len(cur) != prev:
        prev = len(cur)
        u = cur.groupby("user_id").size(); cur = cur[cur["user_id"].isin(u[u>=min_u].index)]
        i = cur.groupby("item_id").size(); cur = cur[cur["item_id"].isin(i[i>=min_i].index)]
    return cur

rev_f = filter_min(rev, MIN_USER_RATINGS, MIN_ITEM_RATINGS)
top_u = rev_f.groupby("user_id").size().sort_values(ascending=False).head(USERS_CAP).index
top_i = rev_f.groupby("item_id").size().sort_values(ascending=False).head(ITEMS_CAP).index
rev_f = rev_f[rev_f["user_id"].isin(top_u) & rev_f["item_id"].isin(top_i)]
rev_f = filter_min(rev_f, MIN_USER_RATINGS, MIN_ITEM_RATINGS)
rev_f = rev_f.sort_values("timestamp").drop_duplicates(["user_id","item_id"], keep="last")
print("Core:", rev_f.shape, "| utenti:", rev_f["user_id"].nunique(), "| item:", rev_f["item_id"].nunique())

# Metadati: il file meta ha campi annidati eterogenei (es. details) che fanno fallire
# il parser pyarrow. Lo leggiamo riga per riga con json, tenendo solo cio' che serve
# e solo per gli item presenti nel core.
import json as _json
from huggingface_hub import HfFileSystem

needed = set(rev_f["item_id"])
meta_path = f"datasets/McAuley-Lab/Amazon-Reviews-2023/raw/meta_categories/meta_{CATEGORY}.jsonl"
fs = HfFileSystem()
mrows = []
with fs.open(meta_path, "rb") as f:
    for raw in f:
        try:
            ex = _json.loads(raw)
        except Exception:
            continue
        pa = ex.get("parent_asin")
        if pa in needed:
            mrows.append({"parent_asin": pa, "title": ex.get("title"), "description": ex.get("description")})
            if len(mrows) >= len(needed):
                break
meta = pd.DataFrame(mrows)
print("Metadati raccolti per", len(meta), "item su", len(needed), "richiesti")
meta.head(3)

## Step 1 — Processamento NLP (title + description)

Pipeline NLTK: minuscolo, rimozione URL/punteggiatura/numeri, tokenizzazione, rimozione stopword, lemmatizzazione. Costruiamo un campo testuale unico per ogni prodotto.

In [ ]:
# description e una lista di stringhe -> uniamo; features (bullet) opzionale
def join_list(x):
    if isinstance(x, (list, np.ndarray)): return " ".join(map(str, x))
    return "" if x is None else str(x)

meta["text_raw"] = meta["title"].fillna("").astype(str) + ". " + meta["description"].apply(join_list)

STOP = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    toks = word_tokenize(t)
    toks = [lemmatizer.lemmatize(w) for w in toks if w not in STOP and len(w) > 2]
    return " ".join(toks)

meta["text_clean"] = meta["text_raw"].apply(clean_text)
print("Esempio prima :", meta["text_raw"].iloc[0][:160])
print("Esempio dopo  :", meta["text_clean"].iloc[0][:160])

In [ ]:
# Teniamo solo gli item presenti nelle recensioni filtrate e con testo non vuoto
item_text = (meta[["parent_asin", "text_clean"]]
             .rename(columns={"parent_asin": "item_id"})
             .dropna(subset=["item_id"]))
item_text = item_text[item_text["text_clean"].str.strip().str.len() > 0]
item_text = item_text.drop_duplicates("item_id").set_index("item_id")

valid_items = sorted(set(rev_f["item_id"]) & set(item_text.index))
item_text = item_text.loc[valid_items]
rev_cb = rev_f[rev_f["item_id"].isin(valid_items)].copy()
print("Item con testo:", len(valid_items), "| recensioni utilizzabili:", len(rev_cb))

ITEM_IDS = list(item_text.index)
ITEM_POS = {it: p for p, it in enumerate(ITEM_IDS)}   # item_id -> riga matrice embedding

## Step 2 — Embedding

**A) Frequenza — TF-IDF** (uni+bigrammi).
**B) Neurale — Transformers** (sentence-transformers `all-MiniLM-L6-v2`): embedding semantici densi a 384 dimensioni.

In [ ]:
# A) TF-IDF
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2)
X_tfidf = tfidf.fit_transform(item_text["text_clean"].tolist())   # sparse [n_item x 5000]
print("TF-IDF:", X_tfidf.shape)

In [ ]:
# B) Transformer
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
X_bert = model.encode(item_text["text_clean"].tolist(), batch_size=64,
                      show_progress_bar=True, normalize_embeddings=True)
X_bert = np.asarray(X_bert)
print("Transformer embedding:", X_bert.shape)

## Step 3 — Predizione rating con K-NN per ogni utente

Approccio content-based: per ogni utente alleniamo un `KNeighborsRegressor` sugli **item che ha gia valutato** (feature = embedding del prodotto, target = rating dato). Prediciamo poi i rating su un insieme di test di item dello stesso utente. La similarita tra prodotti deriva dal **contenuto testuale**, non dalle co-occorrenze di rating.

Valutiamo con uno split train/test per utente e calcoliamo MSE/RMSE globali.

In [ ]:
def evaluate_content_knn(feature_matrix, get_row, ratings_df, k=K_NEIGHBORS,
                          min_ratings=5, test_frac=0.25, seed=RANDOM_STATE):
    # feature_matrix: oggetto indicizzabile per riga; get_row(item_id) -> indice riga
    preds, truths = [], []
    for user, grp in ratings_df.groupby("user_id"):
        items = [it for it in grp["item_id"].tolist() if it in ITEM_POS]
        ratings = grp.set_index("item_id")["rating"]
        if len(items) < min_ratings:
            continue
        tr_items, te_items = train_test_split(items, test_size=test_frac, random_state=seed)
        if len(tr_items) < 1 or len(te_items) < 1:
            continue
        rows_tr = [get_row(it) for it in tr_items]
        rows_te = [get_row(it) for it in te_items]
        X_tr = feature_matrix[rows_tr]; X_te = feature_matrix[rows_te]
        y_tr = ratings.loc[tr_items].values
        n_k = min(k, len(tr_items))
        knn = KNeighborsRegressor(n_neighbors=n_k, metric="cosine", weights="distance")
        knn.fit(X_tr, y_tr)
        p = knn.predict(X_te)
        preds.extend(p.tolist()); truths.extend(ratings.loc[te_items].values.tolist())
    mse = mean_squared_error(truths, preds)
    return {"RMSE": mse ** 0.5, "MSE": mse, "n_pred": len(preds)}

get_row = lambda it: ITEM_POS[it]
res_tfidf = evaluate_content_knn(X_tfidf, get_row, rev_cb)
res_bert  = evaluate_content_knn(X_bert,  get_row, rev_cb)
print("Content-based KNN | TF-IDF      :", {k: round(v,4) for k,v in res_tfidf.items()})
print("Content-based KNN | Transformer :", {k: round(v,4) for k,v in res_bert.items()})

## Step 4 — Valutazione critica: TF-IDF vs Transformer

In [ ]:
emb_cmp = pd.DataFrame([
    {"Embedding": "TF-IDF (frequenza)",   "RMSE": res_tfidf["RMSE"], "MSE": res_tfidf["MSE"]},
    {"Embedding": "Transformer (neurale)", "RMSE": res_bert["RMSE"],  "MSE": res_bert["MSE"]},
])
display(emb_cmp)

ax = emb_cmp.set_index("Embedding")[["RMSE","MSE"]].plot(kind="bar", figsize=(7,4),
        color=["#7AC142","#4A90D9"], rot=0)
ax.set_title("Content-based KNN: TF-IDF vs Transformer")
ax.set_ylabel("Errore")
for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.tight_layout(); plt.show()

**Spunti per il report (TF-IDF vs Transformer):**

- **TF-IDF**: rappresentazione sparsa basata su frequenza dei termini; veloce, interpretabile (si vedono le parole chiave), ma ignora semantica e sinonimi ("cream" vs "lotion" restano distanti) e soffre con descrizioni corte.
- **Transformer**: embedding densi che catturano significato e contesto; migliore su sinonimi/parafrasi, ma piu lento e "black-box". Su descrizioni molto brevi/rumorose il vantaggio si riduce.
- Commentare quale ha dato RMSE/MSE inferiore **sui tuoi dati** e perche (lunghezza testi, ricchezza descrizioni della categoria).

## Step 5 — Valutazione critica: Collaborative Filtering vs Content-Based

Per un confronto **equo**, ricalcoliamo il CF del Progetto Base (K-NN e SVD) sugli **stessi** dati filtrati usati qui.

In [ ]:
from surprise import Dataset, Reader, KNNWithMeans, SVD
from surprise.model_selection import cross_validate

reader = Reader(rating_scale=RATING_SCALE)
data = Dataset.load_from_df(rev_cb[["user_id","item_id","rating"]], reader)

cf_knn = cross_validate(KNNWithMeans(k=K_NEIGHBORS), data, measures=["RMSE","MSE"], cv=5, n_jobs=-1)
cf_svd = cross_validate(SVD(random_state=RANDOM_STATE), data, measures=["RMSE","MSE"], cv=5, n_jobs=-1)

final = pd.DataFrame([
    {"Sistema": "CF — K-NN",            "RMSE": cf_knn["test_rmse"].mean(), "MSE": cf_knn["test_mse"].mean()},
    {"Sistema": "CF — Matrix Fact.(SVD)","RMSE": cf_svd["test_rmse"].mean(), "MSE": cf_svd["test_mse"].mean()},
    {"Sistema": "Content — TF-IDF",      "RMSE": res_tfidf["RMSE"],          "MSE": res_tfidf["MSE"]},
    {"Sistema": "Content — Transformer", "RMSE": res_bert["RMSE"],           "MSE": res_bert["MSE"]},
]).sort_values("RMSE").reset_index(drop=True)
display(final)

ax = final.set_index("Sistema")["RMSE"].plot(kind="barh", figsize=(7,4), color="#9B59B6")
ax.set_title("Confronto finale — RMSE (piu basso = meglio)"); ax.invert_yaxis()
ax.bar_label(ax.containers[0], fmt="%.3f", fontsize=8)
plt.tight_layout(); plt.show()

**Spunti per il report (CF vs Content-Based):**

- **Collaborative Filtering**: sfrutta i pattern di rating della comunita; di solito piu accurato quando c'e abbastanza co-rating, ma soffre il **cold-start** (nuovi utenti/item senza storico) e la sparsita.
- **Content-Based**: usa solo le caratteristiche del prodotto; gestisce il cold-start sugli **item nuovi** (basta il testo) e da raccomandazioni piu spiegabili, ma tende all'**over-specialization** (poca novita) e dipende dalla qualita delle descrizioni.
- Indicare quale ha vinto sui tuoi dati e collegarlo alle caratteristiche della categoria scelta. Spesso un **sistema ibrido** (CF + content) e la soluzione migliore: citalo come conclusione.

---
# PARTE 3 — Progetto Avanzato (Sentiment Analysis)

In [ ]:
import re, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
for pkg in ["stopwords", "wordnet", "omw-1.4", "punkt", "punkt_tab"]:
    try: nltk.download(pkg, quiet=True)
    except Exception: pass

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup OK")

In [ ]:
CATEGORY = "Movies_and_TV"  # stessa categoria dei notebook precedenti
SAMPLE_SIZE = 20000         # subset stratificato per rendere trattabile l'encoding transformer
LLM_SUBSET = 300            # entry per il bonus LLM (evita rate limiting)
LABELS = {0: "negativo", 1: "neutro", 2: "positivo"}

STREAMING = True
MAX_REVIEWS = 400000        # campione iniziale in streaming (poi sottocampioniamo a SAMPLE_SIZE)

## Step 0 — Caricamento recensioni e creazione label di sentiment

Mappiamo i rating: 1-2 -> negativo (0), 3 -> neutro (1), 4-5 -> positivo (2).

In [ ]:
from datasets import load_dataset

def load_reviews(category, max_rows=MAX_REVIEWS, streaming=STREAMING):
    # Le nuove versioni di datasets non eseguono lo script del repo:
    # leggiamo direttamente il file .jsonl grezzo col builder "json".
    url = f"hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/review_categories/{category}.jsonl"
    ds = load_dataset("json", data_files=url, split="train", streaming=streaming)
    if streaming:
        ds = ds.shuffle(seed=42, buffer_size=50000)
        rows = []
        for ex in ds:
            rows.append(ex)
            if len(rows) >= max_rows:
                break
        df_ = pd.DataFrame(rows)
    else:
        df_ = ds.to_pandas()
    # nel jsonl grezzo i campi si chiamano sort_timestamp / helpful_votes
    return df_.rename(columns={"sort_timestamp": "timestamp", "helpful_votes": "helpful_vote"})

rev = load_reviews(CATEGORY)
rev = rev.dropna(subset=["rating"])
rev["rating"] = rev["rating"].astype(float)

def to_sentiment(r):
    if r <= 2: return 0
    if r == 3: return 1
    return 2
rev["sentiment"] = rev["rating"].apply(to_sentiment)

# Campo testuale = titolo + testo della recensione
rev["review_text"] = (rev["title"].fillna("").astype(str) + ". " + rev["text"].fillna("").astype(str))
rev = rev[rev["review_text"].str.strip().str.len() > 3]
print("Recensioni (campione):", len(rev))
print(rev["sentiment"].value_counts().rename(index=LABELS))

In [ ]:
# Distribuzione delle classi (tipicamente molto sbilanciata verso il positivo)
plt.figure(figsize=(6,3.5))
ax = sns.countplot(x=rev["sentiment"].map(LABELS),
                   order=["negativo","neutro","positivo"], palette="RdYlGn")
ax.set_title("Distribuzione del sentiment"); ax.set_xlabel(""); ax.set_ylabel("Conteggio")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Subset stratificato per mantenere le proporzioni delle classi
if len(rev) > SAMPLE_SIZE:
    rev_s, _ = train_test_split(rev, train_size=SAMPLE_SIZE, stratify=rev["sentiment"],
                                random_state=RANDOM_STATE)
else:
    rev_s = rev.copy()
rev_s = rev_s.reset_index(drop=True)
print("Subset di lavoro:", len(rev_s))

## Step 1 — Processamento NLP (title + text)

Pipeline NLTK: minuscolo, rimozione URL/punteggiatura/numeri, tokenizzazione, stopword, lemmatizzazione.

In [ ]:
STOP = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    toks = word_tokenize(t)
    toks = [lemmatizer.lemmatize(w) for w in toks if w not in STOP and len(w) > 2]
    return " ".join(toks)

rev_s["clean"] = rev_s["review_text"].apply(clean_text)
rev_s = rev_s[rev_s["clean"].str.strip().str.len() > 0].reset_index(drop=True)
rev_s[["review_text", "clean", "sentiment"]].head(3)

In [ ]:
# Split train/test stratificato (80/20)
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    rev_s["clean"], rev_s["sentiment"], test_size=0.20,
    stratify=rev_s["sentiment"], random_state=RANDOM_STATE)
print("Train:", len(X_train_txt), "| Test:", len(X_test_txt))

## Step 2 — Embedding

**A) Frequenza — TF-IDF** (matrice sparsa termine-documento).
**B) Neurale — Transformers** (`all-MiniLM-L6-v2`, embedding densi 384-dim).

In [ ]:
# A) TF-IDF
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), min_df=3)
Xtr_tfidf = tfidf.fit_transform(X_train_txt)
Xte_tfidf = tfidf.transform(X_test_txt)
print("TF-IDF:", Xtr_tfidf.shape)

In [ ]:
# B) Transformer
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer("all-MiniLM-L6-v2")
Xtr_bert = np.asarray(emb_model.encode(X_train_txt.tolist(), batch_size=64,
                      show_progress_bar=True, normalize_embeddings=True))
Xte_bert = np.asarray(emb_model.encode(X_test_txt.tolist(), batch_size=64,
                      show_progress_bar=True, normalize_embeddings=True))
print("Transformer:", Xtr_bert.shape)

## Step 3 — Classificazione del sentiment

Classificatori visti in laboratorio: **Decision Tree, Random Forest, Naive Bayes, KNN**. Per Naive Bayes usiamo `MultinomialNB` con TF-IDF (feature non negative) e `GaussianNB` con gli embedding densi del transformer.

In [ ]:
def evaluate_model(model, X_te, y_te):
    y_pred = model.predict(X_te)
    return {
        "acc":  accuracy_score(y_te, y_pred),
        "prec": precision_score(y_te, y_pred, average="macro", zero_division=0),
        "rec":  recall_score(y_te, y_pred, average="macro", zero_division=0),
        "f1":   f1_score(y_te, y_pred, average="macro", zero_division=0),
        "cm":   confusion_matrix(y_te, y_pred),
        "report": classification_report(y_te, y_pred, target_names=list(LABELS.values()), zero_division=0),
    }

def build_models(dense):
    nb = GaussianNB() if dense else MultinomialNB()
    return {
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                                 class_weight="balanced", n_jobs=-1),
        "Naive Bayes":   nb,
        "KNN (k=5)":     KNeighborsClassifier(n_neighbors=5),
    }

In [ ]:
def run_block(Xtr, Xte, dense, tag):
    rows = {}
    for name, model in build_models(dense).items():
        model.fit(Xtr, y_train)
        ev = evaluate_model(model, Xte, y_test)
        rows[name] = ev
        print(f"[{tag}] {name}: acc={ev['acc']:.3f} f1_macro={ev['f1']:.3f}")
    return rows

print("=== Embedding TF-IDF ===")
res_tfidf = run_block(Xtr_tfidf, Xte_tfidf, dense=False, tag="TF-IDF")
print("\n=== Embedding Transformer ===")
res_bert  = run_block(Xtr_bert,  Xte_bert,  dense=True,  tag="BERT")

In [ ]:
# Tabella riepilogativa
def to_table(res, emb):
    return pd.DataFrame([{"Embedding": emb, "Modello": n,
                          "Accuracy": r["acc"], "Precision": r["prec"],
                          "Recall": r["rec"], "F1_macro": r["f1"]} for n, r in res.items()])

summary = pd.concat([to_table(res_tfidf,"TF-IDF"), to_table(res_bert,"Transformer")], ignore_index=True)
display(summary.sort_values("F1_macro", ascending=False).reset_index(drop=True))

In [ ]:
# Grafico F1 macro per modello ed embedding
piv = summary.pivot(index="Modello", columns="Embedding", values="F1_macro")
ax = piv.plot(kind="bar", figsize=(8,4.5), color=["#7AC142","#4A90D9"], rot=20)
ax.set_title("F1 (macro) per classificatore ed embedding"); ax.set_ylabel("F1 macro")
for c in ax.containers: ax.bar_label(c, fmt="%.2f", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix del miglior modello
best_row = summary.sort_values("F1_macro", ascending=False).iloc[0]
best_res = (res_tfidf if best_row["Embedding"]=="TF-IDF" else res_bert)[best_row["Modello"]]
print(f"Migliore: {best_row['Modello']} ({best_row['Embedding']})\n")
print(best_res["report"])

plt.figure(figsize=(5,4))
sns.heatmap(best_res["cm"], annot=True, fmt="d", cmap="Blues",
            xticklabels=list(LABELS.values()), yticklabels=list(LABELS.values()))
plt.title("Confusion Matrix — miglior modello"); plt.xlabel("Predetto"); plt.ylabel("Reale")
plt.tight_layout(); plt.show()

**Spunti per il report (Step 3):**

- Il dataset e fortemente sbilanciato verso il **positivo**: l'accuracy da sola inganna, percio guardiamo **F1 macro** e la confusion matrix (la classe **neutro** e quasi sempre la piu difficile).
- Tipicamente gli **embedding transformer** + Random Forest/SVM danno F1 macro migliore del TF-IDF, perche catturano la semantica; il TF-IDF resta competitivo, velocissimo e interpretabile.
- Discutere l'effetto di `class_weight="balanced"` e l'eventuale uso di tecniche di bilanciamento (oversampling/undersampling).

## Step 4 — BONUS: confronto con metodo basato su LLM (prompting zero-shot)

Classifichiamo `LLM_SUBSET` recensioni del **test set** chiedendo a un LLM di assegnare il sentiment, e confrontiamo l'accuratezza con il miglior modello scikit-learn sugli **stessi** esempi.

Sotto trovi **due opzioni**:
- **Opzione A — API LLM (zero-shot)**: usa una API (OpenAI/Anthropic). Richiede una API key. Approccio "prompting" visto in laboratorio.
- **Opzione B — senza API key**: modello transformer pre-addestrato per il sentiment da HuggingFace (eseguibile su Colab senza chiavi).

Usa quella che preferisci.

In [ ]:
# Sottocampione del TEST set per il bonus (testo grezzo, non pulito: gli LLM preferiscono il testo originale)
test_idx = X_test_txt.index
sub = rev_s.loc[test_idx].sample(n=min(LLM_SUBSET, len(test_idx)), random_state=RANDOM_STATE)
sub_texts = sub["review_text"].tolist()
sub_true  = sub["sentiment"].tolist()
print("Esempi per il bonus:", len(sub_texts))

In [ ]:
# ---- OPZIONE A: API LLM (zero-shot prompting) ----
# Decommenta e inserisci la tua API key. Esempio con OpenAI; per Anthropic vedi nota sotto.
#
# !pip -q install openai
# from openai import OpenAI
# import os, time
# client = OpenAI(api_key="LA_TUA_API_KEY")
#
# SYS = ("Sei un classificatore di sentiment. Rispondi con UNA sola parola tra: "
#        "negativo, neutro, positivo. Recensione 1-2 stelle=negativo, 3=neutro, 4-5=positivo.")
# MAP = {"negativo":0, "neutro":1, "positivo":2}
#
# def llm_classify(text):
#     r = client.chat.completions.create(
#         model="gpt-4o-mini", temperature=0,
#         messages=[{"role":"system","content":SYS},
#                   {"role":"user","content":text[:1500]}])
#     out = r.choices[0].message.content.strip().lower()
#     for k,v in MAP.items():
#         if k in out: return v
#     return 2
#
# llm_pred = []
# for t in sub_texts:
#     try: llm_pred.append(llm_classify(t))
#     except Exception: llm_pred.append(2); time.sleep(1)
#
# Per Anthropic: !pip install anthropic ; client = anthropic.Anthropic(api_key=...) ;
# usa client.messages.create(model="claude-...", max_tokens=5, system=SYS, messages=[...]).
print("Opzione A: decommenta il codice e inserisci la tua API key.")

In [ ]:
# ---- OPZIONE B: transformer pre-addestrato (senza API key) ----
from transformers import pipeline
# Modello che predice 1-5 stelle: mappiamo a negativo/neutro/positivo
star_clf = pipeline("sentiment-analysis",
                    model="nlptown/bert-base-multilingual-uncased-sentiment",
                    truncation=True, max_length=256)

def stars_to_sent(label):
    n = int(label.split()[0])      # es. "4 stars" -> 4
    if n <= 2: return 0
    if n == 3: return 1
    return 2

llm_out = star_clf(sub_texts, batch_size=32)
llm_pred = [stars_to_sent(o["label"]) for o in llm_out]
print("Classificazione LLM/transformer completata.")

In [ ]:
# Confronto: LLM vs miglior modello scikit-learn sugli STESSI esempi
# Predizioni del miglior modello sklearn sul subset
if best_row["Embedding"] == "TF-IDF":
    sub_feat = tfidf.transform(rev_s.loc[sub.index, "clean"])
    best_model = build_models(dense=False)[best_row["Modello"]].fit(Xtr_tfidf, y_train)
else:
    sub_feat = np.asarray(emb_model.encode(rev_s.loc[sub.index, "clean"].tolist(),
                          batch_size=64, normalize_embeddings=True))
    best_model = build_models(dense=True)[best_row["Modello"]].fit(Xtr_bert, y_train)
sk_pred = best_model.predict(sub_feat)

comp = pd.DataFrame({
    "Metodo": [f"scikit-learn ({best_row['Modello']}/{best_row['Embedding']})", "LLM / transformer pre-addestrato"],
    "Accuracy": [accuracy_score(sub_true, sk_pred), accuracy_score(sub_true, llm_pred)],
    "F1_macro": [f1_score(sub_true, sk_pred, average="macro", zero_division=0),
                 f1_score(sub_true, llm_pred, average="macro", zero_division=0)],
})
display(comp)

ax = comp.set_index("Metodo")[["Accuracy","F1_macro"]].plot(kind="bar", figsize=(7,4),
        color=["#4A90D9","#7AC142"], rot=0)
ax.set_title(f"sklearn vs LLM su {len(sub_true)} esempi")
for c in ax.containers: ax.bar_label(c, fmt="%.2f", fontsize=8)
plt.tight_layout(); plt.show()

**Spunti per il report (Bonus LLM):**

- L'LLM/transformer pre-addestrato funziona in **zero-shot** (nessun training sui tuoi dati): comodo quando mancano dati etichettati, ma costo/latenza maggiori e meno controllo.
- Il modello scikit-learn e addestrato sulla **tua** distribuzione: spesso vince sulle classi frequenti, mentre l'LLM puo gestire meglio i casi ambigui/neutri.
- Commentare dove i due metodi sbagliano in modo diverso (es. ironia, recensioni miste) e il trade-off accuratezza / costo / spiegabilita.

## Conclusioni Progetto Avanzato

- **NLP + embedding**: confronto frequenza (TF-IDF) vs neurale (transformer) anche sul task di sentiment.
- **Classificazione**: miglior classificatore + F1 macro + confusion matrix; sottolineare lo sbilanciamento delle classi e la difficolta della classe neutra.
- **Bonus LLM**: confronto critico tra approccio supervisionato classico e LLM zero-shot.

Con i tre notebook (Base, Intermedio, Avanzato) il progetto e completo. Prossimo passo: il **report** che raccoglie metodi, risultati e valutazioni critiche.